# Evaluation
Do retrieval, classification, and citations fail independently?

In [ ]:
import os
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'src'))
import torch
torch.set_num_threads(1)
torch.manual_seed(42)
smoke = os.environ.get('PYTHONIC_SMOKE') == '1'

In [ ]:
from pythonic.agent import Agent, SearchReadClassify
from pythonic.biomedical import EvidencePrediction
from pythonic.data import fixtures
from pythonic.evaluation import evaluate_agent
from pythonic.retrieval import BM25
from pythonic.tools import EvidenceTools
from pythonic.tracing import TraceWriter
sections = [section for paper in fixtures() for section in paper.sections]
tools = EvidenceTools(BM25(sections), sections, lambda q, c, b: EvidencePrediction('yes', 0.7, len(c)), len)
trace = TraceWriter(root / 'artifacts/notebook-trace.jsonl', {'policy': 'fixture'})
report = evaluate_agent(Agent(tools, SearchReadClassify(budget=150), trace=trace), fixtures())
for metric in ('classification', 'retrieval', 'completion_rate', 'citation_provenance_rate', 'task_success'):
    print(metric, report[metric])

In [ ]:
if not smoke:
    import random
    from pythonic.runtime import evidence_agent
    from pythonic.data import load_papers, split_papers
    os.environ.setdefault('HF_HOME', str(root / 'data/hf'))
    checkpoint = root / 'artifacts/evidence/adapters.pt'
    if checkpoint.exists():
        data = root / 'data/pubmedqa.json'
        questions = random.Random(42).sample(list(split_papers(load_papers(data)).test), 12)
        agent = evidence_agent(data, checkpoint, 'cuda' if torch.cuda.is_available() else 'cpu', trace_path=root / 'artifacts/real-trace.jsonl')
        result = evaluate_agent(agent, questions)
        print(result['classification'], result['retrieval'], result['task_success'])

In [ ]:
import json
for name in ('attention', 'generation', 'budgets', 'agent-evaluation'):
    path = root / 'artifacts' / f'{name}.json'
    if path.exists():
        saved = json.loads(path.read_text())
        print(name, saved.get('metadata', {'source_digest': saved.get('source_digest')}))

The fixed fixture classifier gets every citation right but misses two labels. Run evaluate-agent for the full held-out biomedical split.